# CO₂ MMP Prediction — Ahmed (1994) FM Method
### Rathmell Fluid B (SPE 3483) · T = 103°F

Reproduce the Ahmed (1994) first-contact miscibility function to predict the CO₂ minimum
miscibility pressure for Rathmell Fluid B. The method enriches the sample with CO₂ in steps,
tracks the bubble point, and reads off the MMP where the miscibility function FM crosses zero.

## 1. Imports

In [1]:
import numpy as np
from scipy.optimize import brentq

# =============================================================================
# Universal gas constant and PR EOS Omega constants
# =============================================================================
R      = 83.14472    # cm³·bar / (mol·K)
OMA    = 0.45724     # Ωa
OMB    = 0.077796    # Ωb
P_CONV = 0.068948    # psia → bar

# =============================================================================

## 2. Fluid Definition

Rathmell Fluid B from SPE 3483 — 11 components at 103°F.


In [2]:
# Fluid B — Rathmell et al. (1971), SPE 3483
# Reservoir temperature: 103°F = 312.04 K
# =============================================================================
COMP = ['CO2', 'N2', 'C1', 'C2', 'C3', 'iC4', 'nC4', 'iC5', 'nC5', 'C6', 'C7+']
NC   = len(COMP)

T_F = 103.0
T_K = (T_F + 459.67) * 5.0 / 9.0    # 312.04 K

# Feed composition (mol fraction) — Rathmell Table 1
ZI = np.array([3.20, 0.03, 27.81, 8.21, 5.99,
               0.31,  4.10,  1.30,  2.30, 4.62, 42.13]) / 100.0

# Critical temperatures (°F → K)
TC_F = np.array([ 88.79, -232.51, -116.59, 90.104, 205.97,
                 274.91,   305.69,  369.05, 385.61, 453.83, 888.56])
TC   = (TC_F + 459.67) * 5.0 / 9.0

# Critical pressures (psia → bar)
PC_PSIA = np.array([1071.3, 492.31, 667.78, 708.34, 615.76,
                     529.05, 550.66, 491.58, 488.79, 436.62, 253.0])
PC = PC_PSIA * P_CONV

# Acentric factors
ACF = np.array([0.225, 0.040, 0.013, 0.0986, 0.1524,
                0.1848, 0.201,  0.227,  0.251,  0.299, 0.71128])

## 3. EOS Parameter Construction

Builds the PR EOS mixture parameters for a given BIP strategy.
Ahmed (1994) sets α_CO₂ = 1 and uses a temperature-dependent k_{CO₂–HC} = 0.131 + 0.00004·T(°F).


In [3]:
def build_case(case='AHMED'):
    """
    Build PR EOS parameters for Fluid B with the chosen BIP strategy.

    Parameters
    ----------
    case : {'PVTI', 'AHMED_HC', 'AHMED_C7', 'AHMED'}

    Returns
    -------
    AIJ : (NC, NC) ndarray  — cross-parameter matrix  aij = sqrt(ai*aj)*(1-kij)
    BI  : (NC,)   ndarray  — pure-component b values (cm3/mol)
    """
    BIP = np.zeros((NC, NC))

    if   case == 'PVTI':
        D_CO2_HC = 0.10
        D_CO2_C7 = 0.10
    elif case == 'AHMED_HC':
        D_CO2_HC = 0.131 + 0.00004 * T_F
        D_CO2_C7 = 0.10
    elif case == 'AHMED_C7':
        D_CO2_HC = 0.10
        D_CO2_C7 = 0.11
    elif case == 'AHMED':
        D_CO2_HC = 0.131 + 0.00004 * T_F    # 0.1351 at 103 F
        D_CO2_C7 = 0.11
    else:
        raise ValueError(f"Unknown case '{case}'. Choose: PVTI, AHMED_HC, AHMED_C7, AHMED")

    for j in range(2, 10):
        BIP[0, j] = BIP[j, 0] = D_CO2_HC    # CO2 - C1...C6
    BIP[0, 10] = BIP[10, 0] = D_CO2_C7      # CO2 - C7+
    for j in range(2, NC):
        BIP[1, j] = BIP[j, 1] = 0.10        # N2 - HCs
    BIP[0, 1] = BIP[1, 0] = -0.012          # CO2 - N2
    BIP[2, 9]  = BIP[9,  2] = 0.0279        # C1 - C6
    BIP[2, 10] = BIP[10, 2] = 0.054528      # C1 - C7+
    for i in [3, 4]:
        BIP[i, 9]  = BIP[9,  i] = 0.01
        BIP[i, 10] = BIP[10, i] = 0.01      # C2,C3 - C6,C7+

    TR    = T_K / TC
    KAPPA = np.where(
        ACF <= 0.491,
        0.37464 + 1.54226 * ACF - 0.26992 * ACF**2,
        0.379642 + 1.48503 * ACF - 0.164423 * ACF**2 + 0.016666 * ACF**3
    )
    ALPHA = (1.0 + KAPPA * (1.0 - np.sqrt(TR)))**2
    ALPHA[0] = 1.0    # Ahmed (1994): fix alpha_CO2 = 1 for miscibility calcs

    AI  = OMA * R**2 * TC**2 / PC * ALPHA
    BI  = OMB * R * TC / PC
    AIJ = np.outer(np.sqrt(AI), np.sqrt(AI)) * (1.0 - BIP)

    return AIJ, BI


# Quick sanity check
AIJ_check, BI_check = build_case('AHMED')
print(f"Fluid B — {NC} components | T = {T_K:.2f} K ({T_F}°F)")
print(f"ZI sums to: {ZI.sum():.6f}  (should be 1.0)")
print(f"BIP CO2-C1  : {1 - AIJ_check[0,2]/np.sqrt(AIJ_check[0,0]*AIJ_check[2,2]):.4f}  (expected 0.1351)")
print(f"BIP CO2-C7+ : {1 - AIJ_check[0,10]/np.sqrt(AIJ_check[0,0]*AIJ_check[10,10]):.4f}  (expected 0.1100)")

Fluid B — 11 components | T = 312.59 K (103.0°F)
ZI sums to: 1.000000  (should be 1.0)
BIP CO2-C1  : 0.1351  (expected 0.1351)
BIP CO2-C7+ : 0.1100  (expected 0.1100)


## 4. EOS Utilities

In [4]:
# =============================================================================
# EOS Utilities
# =============================================================================

def z_factor(z, P, phase, AIJ, BI):
    am = float(z @ AIJ @ z)
    bm = float(z @ BI)
    A  = am * P / (R * T_K)**2
    B  = bm * P / (R * T_K)
    c  = [1.0, B - 1.0, A - 2*B - 3*B**2, B**3 + B**2 - A*B]
    rt = np.roots(c)
    rr = np.sort(rt[np.abs(rt.imag) < 1e-8].real)
    rr = rr[rr > B + 1e-10]
    if len(rr) == 1:
        return rr[0]
    return rr[0] if phase == 'L' else rr[-1]


def ln_phi(z, P, phase, AIJ, BI):
    am = float(z @ AIJ @ z)
    bm = float(z @ BI)
    A  = am * P / (R * T_K)**2
    B  = bm * P / (R * T_K)
    Z  = z_factor(z, P, phase, AIJ, BI)
    s2 = np.sqrt(2.0)
    da = 2.0 * (AIJ @ z)
    return (
        (BI / bm) * (Z - 1.0)
        - np.log(Z - B)
        - A / (2.0 * s2 * B)
        * (da / am - BI / bm)
        * np.log((Z + (1.0 + s2) * B) / (Z + (1.0 - s2) * B))
    )


# =============================================================================

## 5. Flash Utilities

In [5]:
# Flash Utilities
# =============================================================================

def bubble_flash(z, P, AIJ, BI):
    """Successive substitution flash at bubble point conditions."""
    K = (PC / P) * np.exp(5.373 * (1.0 + ACF) * (1.0 - TC / T_K))
    for _ in range(500):
        y    = K * z;  y /= y.sum()
        Knew = np.exp(ln_phi(z, P, 'L', AIJ, BI) - ln_phi(y, P, 'V', AIJ, BI))
        if np.sum((np.log(Knew / K))**2) < 1e-10:
            break
        K = Knew
    return K, y, float(K @ z)


def bubble_point(z, AIJ, BI):
    """Bracket then refine bubble point pressure (bar) via Brent's method."""
    prev = None
    for p in np.arange(5.0, 800.0, 5.0):
        try:
            val = bubble_flash(z, p, AIJ, BI)[2] - 1.0
            if prev is not None and prev * val < 0:
                return brentq(
                    lambda P: bubble_flash(z, P, AIJ, BI)[2] - 1.0,
                    p - 5.0, p
                )
            prev = val
        except Exception:
            prev = None
    return None


# =============================================================================

## 6. Ahmed FM Utilities

In [6]:
# Ahmed Utilities
# =============================================================================

def ahmed_fm(z, K):
    """Ahmed FM function — sum over all components except C7+ (last index)."""
    terms = [z[i] * (K[i] - 1.0) / K[i]
             for i in range(NC - 1) if K[i] > 1e-8]
    return -np.sum(terms)


def build_fm_curve(AIJ, BI):
    """Sweep CO2 enrichment and return (co2_pct, pb_psia, fm) lists."""
    co2_pct_list, pb_list, fm_list = [], [], []
    for xco2 in np.arange(0.00, 1.51, 0.05):
        Zm    = ZI.copy()
        Zm[0] += xco2
        Zm    /= Zm.sum()
        Pb = bubble_point(Zm, AIJ, BI)
        if Pb is None:
            continue
        K, _, _ = bubble_flash(Zm, Pb, AIJ, BI)
        co2_pct_list.append(Zm[0] * 100.0)
        pb_list.append(Pb / P_CONV)        # bar → psia
        fm_list.append(ahmed_fm(Zm, K))
    return co2_pct_list, pb_list, fm_list


def find_mmp(pb_list, fm_list):
    """Linear interpolation at FM = 0 crossing."""
    for i in range(len(fm_list) - 1):
        if fm_list[i] * fm_list[i + 1] <= 0:
            p1, p2 = pb_list[i], pb_list[i + 1]
            f1, f2 = fm_list[i], fm_list[i + 1]
            return p1 - f1 * (p2 - p1) / (f2 - f1)
    return None


# =============================================================================

## 7. Verification

In [7]:
# Main Driver
# =============================================================================

def run_case(case='AHMED'):
    AIJ, BI                          = build_case(case)
    co2_pct_list, pb_list, fm_list   = build_fm_curve(AIJ, BI)
    MMP                              = find_mmp(pb_list, fm_list)
    return MMP, co2_pct_list, pb_list, fm_list


# Run all four BIP cases
cases   = ['PVTI', 'AHMED_HC', 'AHMED_C7', 'AHMED']
MMP_EXP = 2015.0
results = {}

print(f"{'Case':<14s}  {'MMP (psia)':>10s}  {'Error vs exp':>13s}")
print("-" * 42)
for c in cases:
    mmp, co2_pct, pb_list, fm_list = run_case(c)
    results[c] = (mmp, co2_pct, pb_list, fm_list)
    print(f"{c:<14s}  {mmp:>10.1f}  {abs(mmp - MMP_EXP)/MMP_EXP*100:>12.2f}%")
print("-" * 42)
print(f"{'Experimental':<14s}  {MMP_EXP:>10.1f}  {'reference':>13s}")
print(f"{'Ahmed SPE27032':<14s}  {2053.0:>10.1f}  {abs(2053.0-MMP_EXP)/MMP_EXP*100:>12.2f}%")

Case            MMP (psia)   Error vs exp
------------------------------------------
PVTI                1886.9          6.36%
AHMED_HC            1935.6          3.94%
AHMED_C7            1956.7          2.89%
AHMED               2012.6          0.12%
------------------------------------------
Experimental        2015.0      reference
Ahmed SPE27032      2053.0          1.89%


## 8. Results

In [8]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

mmp_pred, co2_pct, pb_psia, fm = results['AHMED']
mmp_exp   = 2015.0
mmp_ahmed = 2053.0
error_pct = abs(mmp_pred - mmp_exp) / mmp_exp * 100

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Bubble Point Increase During CO2 Enrichment",
        "Ahmed Miscibility Function  (FM = 0 -> MMP)"
    ),
    horizontal_spacing=0.10
)

fig.add_trace(go.Scatter(
    x=co2_pct, y=pb_psia, mode="lines+markers", name="Bubble Point",
    line=dict(color="#1f77b4"),
    hovertemplate="CO2 = %{x:.1f} mol%<br>Pb = %{y:.0f} psia<extra></extra>"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=pb_psia, y=fm, mode="lines+markers", name="FM (excl. C7+)",
    line=dict(color="#d62728"),
    hovertemplate="Pb = %{x:.0f} psia<br>FM = %{y:.4f}<extra></extra>"
), row=1, col=2)

fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1.5, row=1, col=2)
fig.add_vline(x=mmp_pred,  line_dash="dash",    line_color="#2ca02c",    line_width=2, row=1, col=2)
fig.add_vline(x=mmp_exp,   line_dash="dot",     line_color="black",      line_width=2, row=1, col=2)
fig.add_vline(x=mmp_ahmed, line_dash="dashdot", line_color="darkorange", line_width=2, row=1, col=2)

fig.add_trace(go.Scatter(
    x=[mmp_pred], y=[0], mode="markers",
    marker=dict(size=12, color="#2ca02c", symbol="star"),
    name="Predicted MMP",
    hovertemplate=f"Predicted MMP = {mmp_pred:.1f} psia<extra></extra>"
), row=1, col=2)

fig.add_annotation(x=mmp_pred, y=0.18,
    text=f"<b>Our prediction</b><br>{mmp_pred:.1f} psia",
    showarrow=True, arrowhead=2, ax=-65, ay=-35,
    font=dict(color="#2ca02c"), xref="x2", yref="y2")

fig.add_annotation(x=mmp_exp, y=0.42,
    text=f"<b>Slimtube exp.</b><br>{mmp_exp:.1f} psia",
    showarrow=True, arrowhead=2, ax=55, ay=-50,
    font=dict(color="black"), xref="x2", yref="y2")

fig.add_annotation(x=mmp_ahmed, y=0.65,
    text=f"<b>Ahmed (1994)</b><br>{mmp_ahmed:.0f} psia",
    showarrow=True, arrowhead=2, ax=45, ay=-40,
    font=dict(color="darkorange"), xref="x2", yref="y2")

fig.update_xaxes(title_text="CO2 in Mixture (mol%)", row=1, col=1)
fig.update_yaxes(title_text="Bubble Point Pressure (psia)", row=1, col=1)
fig.update_xaxes(title_text="Bubble Point Pressure (psia)", range=[1500, 2600], row=1, col=2)
fig.update_yaxes(title_text="FM (excluding C7+)", range=[-0.12, 1.45], row=1, col=2)

fig.update_layout(
    template="plotly_white",
    width=1500, height=700,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    title=dict(
        text=(
            "Ahmed (1994) MMP Prediction - Rathmell Fluid B at 103°F"
            "<br>"
            f"<sup>Predicted MMP = {mmp_pred:.1f} psia  |  "
            f"Experimental = {mmp_exp:.1f} psia  |  "
            f"Error = {error_pct:.2f}%  |  "
            f"Ahmed (1994) = {mmp_ahmed:.0f} psia</sup>"
        ),
        x=0.5
    )
)

fig.show()


## 9. Comparison

| Source | MMP (psia) | Error |
|--------|-----------|-------|
| Rathmell et al. (1971) slimtube | 2015.0 | — |
| Ahmed (1994) SPE 27032 | 2053.0 | +1.9% |
| This work — AHMED BIPs | 2012.6 | −0.1% |


In [9]:
# =============================================================================
# 8. Comparison — Rathmell (1971) and Ahmed (1994)
# =============================================================================
import pandas as pd

data = {
    'Source'          : ['Rathmell et al. (1971) — slimtube',
                         'Ahmed (1994) — SPE 27032',
                         'This work — PVTI BIPs',
                         'This work — Ahmed HC BIPs',
                         'This work — Ahmed C7+ BIP',
                         'This work — Ahmed full BIPs'],
    'Method'          : ['Slimtube experiment',
                         'FM analytical (PR EOS)',
                         'FM analytical (PR EOS)',
                         'FM analytical (PR EOS)',
                         'FM analytical (PR EOS)',
                         'FM analytical (PR EOS)'],
    'MMP (psia)'      : [2015.0, 2053.0,
                         results['PVTI'][0],
                         results['AHMED_HC'][0],
                         results['AHMED_C7'][0],
                         results['AHMED'][0]],
    'Error vs exp (%)': [0.00,
                         round(abs(2053.0  - 2015.0) / 2015.0 * 100, 2),
                         round(abs(results['PVTI'][0]     - 2015.0) / 2015.0 * 100, 2),
                         round(abs(results['AHMED_HC'][0] - 2015.0) / 2015.0 * 100, 2),
                         round(abs(results['AHMED_C7'][0] - 2015.0) / 2015.0 * 100, 2),
                         round(abs(results['AHMED'][0]    - 2015.0) / 2015.0 * 100, 2)],
}

df = pd.DataFrame(data)
df['MMP (psia)'] = df['MMP (psia)'].map('{:.1f}'.format)
print(df.to_string(index=False))


                           Source                 Method MMP (psia)  Error vs exp (%)
Rathmell et al. (1971) — slimtube    Slimtube experiment     2015.0              0.00
         Ahmed (1994) — SPE 27032 FM analytical (PR EOS)     2053.0              1.89
            This work — PVTI BIPs FM analytical (PR EOS)     1886.9              6.36
        This work — Ahmed HC BIPs FM analytical (PR EOS)     1935.6              3.94
        This work — Ahmed C7+ BIP FM analytical (PR EOS)     1956.7              2.89
      This work — Ahmed full BIPs FM analytical (PR EOS)     2012.6              0.12
